In [ ]:
# Prism RAG — end-to-end smoke test
#
# Что проверяем:
# 1. SONAR грузится и считает эмбеддинги (1024-dim, L2-norm)
# 2. Qdrant принимает данные (dense vectors + lexical payload)
# 3. LLM-клиент (OpenAI) делает structured extraction по чанкам
# 4. Поиск возвращает релевантные параграфы

import logging
import os

from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY must be set in .env"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# Build the full Prism stack: SONAR + Qdrant + LLM
from qdrant_client import AsyncQdrantClient

from prism import (
    LLMClient,
    MarkdownChunker,
    Prism,
    PrismGraph,
    QdrantBackend,
    SonarEmbedder,
)

qdrant_url = os.getenv("QDRANT_URL", "http://localhost:6333")

embedder = SonarEmbedder(source_lang="rus_Cyrl", device=os.getenv("PRISM_SONAR_DEVICE", "cpu"))
qdrant = QdrantBackend(AsyncQdrantClient(url=qdrant_url), collection_name="prism_smoke")

graph = await PrismGraph.create(qdrant, embedder, recreate=True)
# Reasoning-style models (gpt-5.x) ignore custom temperature — pass None to skip it.
llm = LLMClient(temperature=None)
prism = Prism(graph, llm, chunker=MarkdownChunker(max_tokens=256, min_section_tokens=10))
print("stack ready")

In [ ]:
# Ingest a small Russian corpus and search it
sample_md = """# Архитектура Prism

Prism — это RAG-движок, который раскладывает запрос пользователя в набор атомарных ключевых фраз и ищет каждую из них по гибридному индексу.

## Хранилища

Векторы и лексический payload хранятся в одном Qdrant с HNSW-настройкой m=256, ef_construct=512. Поверх dense-каналу работает payload-фильтр по стемам Snowball, дающий строгое AND с проверкой proximity внутри предложения.

## Эмбеддинги

Текст кодируется моделью Meta SONAR. Это 1024-мерные мультиязычные эмбеддинги, поддерживающие 200 языков через коды FLORES-200. Все векторы L2-нормализованы, поэтому косинус сводится к скалярному произведению.

## Поиск

Запрос сначала разбирается LLM на ключевые фразы и синонимы. Каждая фраза эмбеддится отдельно, плюс считается агрегатный вектор как среднее. По каждому вектору идёт независимый Qdrant-поиск с динамическим порогом по перцентилю распределения скоров — это устойчивее фиксированного порога косинуса.

# Гибридное ранжирование

После векторного поиска подключается лексический канал — AND по стемам ключевой фразы с post-фильтром на уровне предложения. Результаты объединяются, дедуплицируются, затем LLM-судья отфильтровывает нерелевантные параграфы перед тем как вернуть их клиенту.
"""

nodes = await prism.ingest(sample_md)
print(f"ingested {len(nodes)} nodes")
for n in nodes:
    print(f"  - {n.name}  (keypoints: {n.keypoints[:3]}...)")

In [ ]:
# Query
result = await prism.search("какие модели эмбеддингов используются и почему 1024 размерность?")

print("query:    ", result.query)
print("keypoints:", result.keypoints)
print("note:     ", result.note)
print(f"\n--- {len(result.paragraphs)} paragraph(s) ---\n")
for i, p in enumerate(result.paragraphs, 1):
    print(f"[{i}] {p}\n")

In [ ]:
# Cleanup
await qdrant.client.close()
print("done")